# NLP, LLM & Generative AI - Practice Exercises
## NELIREF 3rd Data Science & AI Summer School
### Task 2 Companion Notebook

**Instructor:** Joseph L. Tsenum

**Due Date:** Wednesday, July 8, 2026 (11:59 PM EDT)

**Level:** Introductory (core Python only - no external NLP libraries required)

**Goal:** Build the core NLP/LLM building blocks by hand: tokenization, bag-of-words,
preprocessing, embeddings similarity, and a mini language model.

---

### How to use this notebook
1. Read the markdown cell above each exercise.
2. Fill in the code where you see `# TODO`.
3. Run the **Test** cell right below your solution — it will tell you if you passed.
4. Add a short comment (1-2 lines) explaining what you learned, just like in Task 1.
5. Do **not** delete the test cells — they are part of your grade (Practice Exercises, 40 pts).

> Tip: If you get stuck, ask your AI assistant (Copilot/Claude/ChatGPT) to explain the
> error message or suggest an approach — but make sure *you* understand the final answer.

---


## Exercise 1: Build a Tokenizer

**Concept:** In Part 3 of `NELIREF_Basics_NLP_LLM_GenAI.ipynb`, we saw that raw text has to be
broken into **tokens** before any model can process it.

**Your task:** Write a function `tokenize(text)` that:
- Splits `text` on whitespace
- Lowercases every token
- Strips punctuation (`.`, `,`, `!`, `?`, `;`, `:`, `"`, `'`) from the start/end of each token
- Returns a list of tokens (drop any empty strings that result)

**Example:**
```
tokenize("Apple was founded by Steve Jobs.")
-> ['apple', 'was', 'founded', 'by', 'steve', 'jobs']
```


In [1]:
import string

def tokenize(text):
    """Split text into a list of clean, lowercase tokens."""
    # Clean each whitespace-separated word before keeping it.
    tokens = []
    for token in text.split():
        clean_token = token.lower().strip(string.punctuation)
        if clean_token:
            tokens.append(clean_token)
    return tokens

# quick manual check
print(tokenize("Apple was founded by Steve Jobs."))

['apple', 'was', 'founded', 'by', 'steve', 'jobs']


In [2]:
# Test Cell: Exercise 1 (do not modify)
result = tokenize("Apple was founded by Steve Jobs.")
assert result == ['apple', 'was', 'founded', 'by', 'steve', 'jobs'], f"Got {result}"

result2 = tokenize("Hello, world!! NLP is FUN.")
assert result2 == ['hello', 'world', 'nlp', 'is', 'fun'], f"Got {result2}"

print("Exercise 1 PASSED \u2705")


Exercise 1 PASSED ✅


**What did you learn?**

Tokenization breaks text into smaller words that a computer can process. Lowercasing and removing punctuation makes matching the same word more consistent.

## Exercise 2: Bag-of-Words Word Counter

**Concept:** Part 4 covered representing text as numbers. The simplest representation is
**Bag-of-Words (BoW)** — just count how many times each token appears.

**Your task:** Write a function `bag_of_words(tokens)` that takes a list of tokens (use your
`tokenize()` from Exercise 1!) and returns a dictionary mapping `token -> count`.

Then write `top_n_words(bow, n)` that returns the `n` most frequent words as a list of
`(word, count)` tuples, sorted from most to least frequent.


In [3]:
def bag_of_words(tokens):
    """Return a dict mapping each token to how many times it appears."""
    # Count each token so the text becomes a simple numeric representation.
    counts = {}
    for token in tokens:
        counts[token] = counts.get(token, 0) + 1
    return counts

def top_n_words(bow, n=3):
    """Return the n most frequent (word, count) pairs, sorted descending by count."""
    # Sort word-count pairs from the largest count to the smallest.
    top = sorted(bow.items(), key=lambda item: item[1], reverse=True)
    return top[:n]

sample_text = "the cat sat on the mat the cat was happy"
tokens = tokenize(sample_text)
bow = bag_of_words(tokens)
print(bow)
print(top_n_words(bow, 3))

{'the': 3, 'cat': 2, 'sat': 1, 'on': 1, 'mat': 1, 'was': 1, 'happy': 1}
[('the', 3), ('cat', 2), ('sat', 1)]


In [4]:
# Test Cell: Exercise 2 (do not modify)
tokens = tokenize("the cat sat on the mat the cat was happy")
bow = bag_of_words(tokens)
assert bow.get("the") == 3, f"Expected 'the' to appear 3 times, got {bow.get('the')}"
assert bow.get("cat") == 2, f"Expected 'cat' to appear 2 times, got {bow.get('cat')}"

top = top_n_words(bow, 2)
assert top[0][0] == "the" and top[0][1] == 3, f"Got {top}"

print("Exercise 2 PASSED \u2705")


Exercise 2 PASSED ✅


**What did you learn?**

A bag of words ignores word order and records how often each word appears. Sorting the counts helps us find the most common words in a piece of text.

## Exercise 3: Text Preprocessing Pipeline (Stopword Removal)

**Concept:** Part 3 also introduced **preprocessing** — removing common "filler" words
(stopwords) like *the, is, a, of* that carry little meaning on their own.

**Your task:**
1. Use the `STOPWORDS` set given below.
2. Write `remove_stopwords(tokens)` that filters out any token in `STOPWORDS`.
3. Write `preprocess(text)` that combines `tokenize()` + `remove_stopwords()` into a single
   pipeline function.


In [5]:
STOPWORDS = {"the", "a", "an", "is", "was", "were", "on", "in", "at", "of", "to", "and", "by"}

def remove_stopwords(tokens):
    """Return tokens with any word in STOPWORDS removed."""
    # Remove common words so the more informative words stand out.
    filtered = [token for token in tokens if token not in STOPWORDS]
    return filtered

def preprocess(text):
    """Full pipeline: tokenize the text, then remove stopwords."""
    # Apply the cleaning steps in the same order as an NLP pipeline.
    return remove_stopwords(tokenize(text))

print(preprocess("Apple was founded by Steve Jobs in the United States."))

['apple', 'founded', 'steve', 'jobs', 'united', 'states']


In [6]:
# Test Cell: Exercise 3 (do not modify)
result = preprocess("Apple was founded by Steve Jobs in the United States.")
assert "was" not in result and "the" not in result and "in" not in result, f"Stopwords not removed: {result}"
assert "apple" in result and "jobs" in result, f"Got {result}"

print("Exercise 3 PASSED \u2705")


Exercise 3 PASSED ✅


**What did you learn?**

A preprocessing pipeline applies several cleaning steps in order. Removing stopwords can reduce unimportant noise while keeping the main meaning-bearing words.

## Exercise 4: Cosine Similarity for Word Embeddings

**Concept:** Part 4 showed toy word embeddings (vectors) where similar words point in similar
directions. **Cosine similarity** measures how similar two vectors' directions are:

$$\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\|A\| \times \|B\|}$$

A value close to **1** means very similar, close to **0** means unrelated, and close to
**-1** means opposite.

**Your task:** Using only core Python (no numpy required, though you may use it if you
prefer), write:
- `dot_product(v1, v2)`
- `magnitude(v)`
- `cosine_similarity(v1, v2)`

Then use the toy embeddings below to compare `king` vs `queen` and `king` vs `cat`, and
confirm that `king`/`queen` are more similar than `king`/`cat`.


In [7]:
import math

word_embeddings = {
    'king':   [0.9,  0.1, -0.3],
    'queen':  [0.85, 0.1, -0.4],
    'man':    [0.8,  0.2,  0.3],
    'woman':  [0.8,  0.2, -0.3],
    'prince': [0.75, 0.05, -0.2],
    'cat':    [0.2,  0.8,  0.5],
    'dog':    [0.25, 0.8,  0.4],
}

def dot_product(v1, v2):
    """Return the dot product of two equal-length vectors."""
    # Multiply matching values and add the products together.
    return sum(value1 * value2 for value1, value2 in zip(v1, v2))

def magnitude(v):
    """Return the Euclidean length (magnitude) of a vector."""
    # The magnitude is the square root of the sum of squared values.
    return math.sqrt(sum(value * value for value in v))

def cosine_similarity(v1, v2):
    """Return the cosine similarity between two vectors."""
    # Compare direction, while guarding against a vector with no length.
    denominator = magnitude(v1) * magnitude(v2)
    if denominator == 0:
        return 0.0
    return dot_product(v1, v2) / denominator

king_queen = cosine_similarity(word_embeddings['king'], word_embeddings['queen'])
king_cat = cosine_similarity(word_embeddings['king'], word_embeddings['cat'])

print(f"king vs queen similarity: {king_queen:.3f}")
print(f"king vs cat similarity:   {king_cat:.3f}")

king vs queen similarity: 0.993
king vs cat similarity:   0.120


In [8]:
# Test Cell: Exercise 4 (do not modify)
same = cosine_similarity([1, 0], [1, 0])
assert abs(same - 1.0) < 1e-6, f"Identical vectors should have similarity 1.0, got {same}"

orth = cosine_similarity([1, 0], [0, 1])
assert abs(orth - 0.0) < 1e-6, f"Perpendicular vectors should have similarity 0.0, got {orth}"

king_queen = cosine_similarity(word_embeddings['king'], word_embeddings['queen'])
king_cat = cosine_similarity(word_embeddings['king'], word_embeddings['cat'])
assert king_queen > king_cat, "king/queen should be MORE similar than king/cat"

print("Exercise 4 PASSED \u2705")


Exercise 4 PASSED ✅


**What did you learn?**

Cosine similarity compares the direction of two vectors instead of just their size. Vectors pointing in similar directions have a value closer to 1, which is why king and queen are more similar here than king and cat.

## Exercise 5: Mini Bigram Language Model (Next-Word Predictor)

**Concept:** Part 6 explained that a **language model** predicts the probability of the next
word given previous words. Let's build the simplest possible version: a **bigram model**
that, for every word, remembers which words followed it in a training corpus.

**Your task:**
1. Write `build_bigram_model(tokens)` that returns a dictionary mapping
   `word -> list of words that followed it anywhere in the corpus` (duplicates allowed —
   more common follow-up words should appear more often in the list).
2. Write `predict_next_word(word, model)` that returns the **most frequent** next word for
   `word` (use `collections.Counter` or your own counting logic). Return `None` if the word
   is not in the model.
3. Write `generate_text(seed_word, model, n_words=5)` that starts at `seed_word` and
   repeatedly calls `predict_next_word` to generate a short sequence of `n_words` words.


In [9]:
from collections import Counter

corpus = "the cat sat on the mat the dog sat on the floor the cat ran to the door"
corpus_tokens = tokenize(corpus)

def build_bigram_model(tokens):
    """Return dict: word -> list of words that immediately followed it in tokens."""
    # Store every neighboring pair so repeated pairs keep their frequency.
    model = {}
    for current_word, next_word in zip(tokens, tokens[1:]):
        model.setdefault(current_word, []).append(next_word)
    return model

def predict_next_word(word, model):
    """Return the single most frequent word that follows `word` in the model."""
    # The most common follower is the model's simplest prediction.
    if word not in model:
        return None
    return Counter(model[word]).most_common(1)[0][0]

def generate_text(seed_word, model, n_words=5):
    """Generate a sequence of n_words starting from seed_word using the bigram model."""
    # Continue from each prediction until the requested length or an unknown word.
    sequence = [seed_word]
    while len(sequence) < n_words:
        next_word = predict_next_word(sequence[-1], model)
        if next_word is None:
            break
        sequence.append(next_word)
    return sequence

bigram_model = build_bigram_model(corpus_tokens)
print("Next word after 'the':", predict_next_word("the", bigram_model))
print("Generated text:", " ".join(generate_text("the", bigram_model, 5)))

Next word after 'the': cat
Generated text: the cat sat on the


In [10]:
# Test Cell: Exercise 5 (do not modify)
bigram_model = build_bigram_model(corpus_tokens)
assert "cat" in bigram_model.get("the", []), f"Expected 'cat' to follow 'the' at least once, got {bigram_model.get('the')}"

next_word = predict_next_word("the", bigram_model)
assert next_word is not None, "predict_next_word should not return None for a word in the corpus"

assert predict_next_word("zzz_not_a_word", bigram_model) is None, "Unknown words should return None"

generated = generate_text("the", bigram_model, 5)
assert len(generated) >= 1 and generated[0] == "the", f"Got {generated}"

print("Exercise 5 PASSED \u2705")


Exercise 5 PASSED ✅


**What did you learn?**

A bigram model learns which word commonly follows each word. It can generate text one word at a time, but its predictions are limited because it only remembers one previous word.

## Summary Check

Run the cell below once all 5 exercises pass — it re-runs every test and gives you a
final scorecard before you commit and submit.


In [11]:
print("="*60)
print("PRACTICE EXERCISES - FINAL CHECK")
print("="*60)

results = {}

try:
    assert tokenize("Apple was founded by Steve Jobs.") == ['apple', 'was', 'founded', 'by', 'steve', 'jobs']
    results["Exercise 1: Tokenizer"] = "PASS"
except Exception as e:
    results["Exercise 1: Tokenizer"] = f"FAIL ({e})"

try:
    bow = bag_of_words(tokenize("the cat sat on the mat the cat was happy"))
    assert bow.get("the") == 3 and bow.get("cat") == 2
    results["Exercise 2: Bag-of-Words"] = "PASS"
except Exception as e:
    results["Exercise 2: Bag-of-Words"] = f"FAIL ({e})"

try:
    result = preprocess("Apple was founded by Steve Jobs in the United States.")
    assert "was" not in result and "apple" in result
    results["Exercise 3: Preprocessing"] = "PASS"
except Exception as e:
    results["Exercise 3: Preprocessing"] = f"FAIL ({e})"

try:
    kq = cosine_similarity(word_embeddings['king'], word_embeddings['queen'])
    kc = cosine_similarity(word_embeddings['king'], word_embeddings['cat'])
    assert kq > kc
    results["Exercise 4: Cosine Similarity"] = "PASS"
except Exception as e:
    results["Exercise 4: Cosine Similarity"] = f"FAIL ({e})"

try:
    model = build_bigram_model(corpus_tokens)
    assert predict_next_word("the", model) is not None
    results["Exercise 5: Bigram Model"] = "PASS"
except Exception as e:
    results["Exercise 5: Bigram Model"] = f"FAIL ({e})"

for k, v in results.items():
    mark = "\u2705" if v == "PASS" else "\u274c"
    print(f"{mark} {k}: {v}")

passed = sum(1 for v in results.values() if v == "PASS")
print("\n" + "="*60)
print(f"SCORE: {passed}/5 exercises passing")
print("="*60)


PRACTICE EXERCISES - FINAL CHECK
✅ Exercise 1: Tokenizer: PASS
✅ Exercise 2: Bag-of-Words: PASS
✅ Exercise 3: Preprocessing: PASS
✅ Exercise 4: Cosine Similarity: PASS
✅ Exercise 5: Bigram Model: PASS

SCORE: 5/5 exercises passing


---

## (Optional) Bonus: Mini Rule-Based Chatbot / Attention Visualizer

See **Part 4** of the Task 2 assignment document for full requirements. Build this in a
**separate** notebook called `my_nlp_project.ipynb` — don't add it here.

## References

- Vaswani, A., et al. (2017). *Attention Is All You Need.*
- Devlin, J., et al. (2019). *BERT: Pre-training of Deep Bidirectional Transformers.*
- Mikolov, T., et al. (2013). *Efficient Estimation of Word Representations in Vector Space* (Word2Vec).
- See also: `NELIREF_Basics_NLP_LLM_GenAI.ipynb` (concept notebook) and the Task 2 slide deck.

*Created for NELIREF 3rd Data Science & AI Summer School 2026*
